## Parallel Chain - Custom runnable

In [18]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableSequence, RunnableLambda, RunnableParallel


# 1. Explicitly load the variables from your .env file
load_dotenv()

# 2. (Optional) Run a quick sanity print statement to check it loaded
print("Loaded Key prefix:", os.environ.get("OPENAI_API_KEY")[:6])

# 3. Initialize your LangChain architecture
llm_openai = init_chat_model(model="gpt-4o-mini", model_provider="openai", temperature=0)

Loaded Key prefix: sk-pro


In [19]:
#Task 1

prompt_template = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a movie reviewer "),
        ("user",  "Please summarize the moview in brief : {input}")
    ]
)

In [20]:
# Task 2 [LLM]

llm_openai = init_chat_model(model="gpt-4o-mini", model_provider="openai", temperature=0)

In [21]:
#Task 3 [String Parser]

str_parser = StrOutputParser()

In [22]:
#Custom task for parsing the output into dictionary

def dictionary_maker(text:str)->dict:
        return {"text" : text}

dictionary_maker_runnable = RunnableLambda(dictionary_maker)

### Parallel Chain 1

In [23]:
#Task 1 - Chain 1

linkedin_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a linkedin post generator. Make sure the response is a single paragraph"),
        ("user",  "Create a post for the following text : {text}")
    ]
)

#Task 2 - LLM

llm_openai = init_chat_model(model="gpt-4o-mini", model_provider="openai", temperature=0)

# Task 3 - String parser

str_parser = StrOutputParser()

chain_linkedin = linkedin_prompt | llm_openai | str_parser


## Parallel Chain 2

In [24]:
def insta_chain(text:dict):
    text = text["text"]
    #Task 1 - Chain 2

    insta_prompt = ChatPromptTemplate.from_messages(
        [
            ("system", "You are a insta post generator "),
            ("user",  "Create a post for the following text : {text}")
        ]
    )

    #Task 2 - LLM

    llm_openai = init_chat_model(model="gpt-4o-mini", model_provider="openai", temperature=0)

    # Task 3 - String parser

    str_parser = StrOutputParser()

    chain_insta = insta_prompt | llm_openai | str_parser

    result = chain_insta.invoke(text)

    return result

insta_chain_runnable = RunnableLambda(insta_chain)


## Final Orchestration

In [25]:
final_chain = (
                prompt_template |
                llm_openai | 
                str_parser |
                dictionary_maker_runnable |
                RunnableParallel(branches = {"linkedin": chain_linkedin, "instagram": insta_chain_runnable})
)

In [26]:
final_chain.invoke("Pushpa")

{'branches': {'linkedin': '🚀 Excited to share my thoughts on "Pushpa: The Rise," a gripping 2021 Telugu action drama directed by Sukumar! 🌟 Starring the incredibly talented Allu Arjun as Pushpa Raj, the film takes us on a thrilling journey through the world of red sandalwood smuggling in Andhra Pradesh. With its intense narrative, powerful performances, and a captivating score by Devi Sri Prasad, "Pushpa" brilliantly explores themes of ambition, betrayal, and survival. The chemistry between Allu Arjun and Rashmika Mandanna adds depth to the story, making it a must-watch! 🎬 The film\'s success has sparked discussions about a sequel, and I can\'t wait to see where Pushpa\'s journey takes us next! #Pushpa #AlluArjun #TeluguCinema #FilmReview #ActionDrama',
  'instagram': '🌟🎬 **Movie Spotlight: Pushpa: The Rise** 🎬🌟\n\nGet ready to dive into the gripping world of "Pushpa: The Rise"! 🚀✨ Directed by the talented Sukumar, this 2021 Telugu action drama stars the incredible Allu Arjun as Pushpa

### Chain as a Runnable

In [27]:
def beautify(final_chain_response:dict)->dict:
    linkedin_response = final_chain_response['branches']['linkedin']
    insta_response = final_chain_response['branches']['instagram']

    return{"linkedin":linkedin_response, "instagram":insta_response}

beautify_runnable = RunnableLambda(beautify)


beautified_chain = final_chain | beautify_runnable

beautified_chain.invoke("Hollow Man 2")


{'linkedin': '🎬 Excited to dive into the thrilling world of "Hollow Man 2"! This 2006 science fiction horror film takes us on a gripping journey with Michael Griffin, a former military operative who becomes the unwitting subject of a groundbreaking invisibility experiment. As he grapples with the dark side effects of his newfound powers, the film masterfully explores themes of power, isolation, and the ethical boundaries of scientific experimentation. With suspenseful action and chilling horror elements, "Hollow Man 2" serves as a stark reminder of the chaos that can ensue when humanity pushes the limits of science. Have you seen it? What are your thoughts on the moral implications of such experiments? #HollowMan2 #ScienceFiction #FilmDiscussion #EthicsInScience',
 'instagram': '🎬✨ Dive into the chilling world of "Hollow Man 2"! 🌌👻 Released in 2006, this sci-fi horror sequel takes us on a wild ride with Michael Griffin, a former military operative turned invisible due to a dangerous ex